# Exercise 1: Building an MCP Server

**Level:** Basic

In this exercise, you will build a **Model Context Protocol (MCP)** server from scratch. MCP is an open standard that lets LLMs connect to external tools and data sources through a unified interface.

**What you will learn:**
- What MCP is and why it matters for agent development
- How to define tools using `@mcp.tool()`
- How to expose resources via `@mcp.resource()`
- How to test your MCP server locally

## 1. Setup & Installation

In [ ]:
!pip install "mcp[cli]" httpx -q

## 2. What is MCP?

The **Model Context Protocol** provides a standardized way for AI models to:
- Call **tools** (functions that perform actions)
- Read **resources** (data the model can access)
- Use **prompts** (reusable prompt templates)

Think of it as a USB-C port for AI — one standard interface that connects to many different capabilities.

```
LLM <---> MCP Client <---> MCP Server <---> Tools / Data / APIs
```

## 3. Creating Your First MCP Server

We will write MCP server code to files and run them. In Colab, we cannot run a persistent server directly in a cell, so we will write the server as a script and test its components.

In [ ]:
%%writefile mcp_server.py
"""A simple MCP server with tools and resources."""

from mcp.server.fastmcp import FastMCP

# Create the MCP server instance
mcp = FastMCP("TrainingServer")


# --- TOOLS ---
# Tools are functions the LLM can call to perform actions.

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b


@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b


@mcp.tool()
def get_weather(city: str) -> dict:
    """Get current weather for a city (simulated)."""
    # In a real server, this would call a weather API
    weather_data = {
        "istanbul": {"temp": 22, "condition": "Sunny", "humidity": 45},
        "london": {"temp": 15, "condition": "Cloudy", "humidity": 70},
        "tokyo": {"temp": 28, "condition": "Humid", "humidity": 80},
        "new york": {"temp": 20, "condition": "Clear", "humidity": 55},
    }
    city_lower = city.lower()
    if city_lower in weather_data:
        return {"city": city, **weather_data[city_lower]}
    return {"city": city, "error": "City not found in database"}


# --- RESOURCES ---
# Resources provide data that the LLM can read.

@mcp.resource("config://app")
def get_app_config() -> str:
    """Return application configuration."""
    return "App Name: TrainingServer\nVersion: 1.0\nEnvironment: development"


@mcp.resource("data://cities")
def get_supported_cities() -> str:
    """Return list of supported cities."""
    return "Supported cities: Istanbul, London, Tokyo, New York"


if __name__ == "__main__":
    mcp.run()

## 4. Understanding Tool Definitions

Let's inspect what MCP sees when it looks at our tools. The `@mcp.tool()` decorator automatically extracts:
- The function name → tool name
- The docstring → tool description
- The type hints → parameter schema

In [ ]:
# Import and inspect our server's tools directly
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("InspectionServer")

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@mcp.tool()
def get_weather(city: str) -> dict:
    """Get current weather for a city (simulated)."""
    weather_data = {
        "istanbul": {"temp": 22, "condition": "Sunny", "humidity": 45},
        "london": {"temp": 15, "condition": "Cloudy", "humidity": 70},
    }
    city_lower = city.lower()
    if city_lower in weather_data:
        return {"city": city, **weather_data[city_lower]}
    return {"city": city, "error": "City not found"}

# List registered tools
print("Registered tools:")
for name, tool in mcp._tool_manager._tools.items():
    print(f"  - {name}: {tool.description}")

## 5. Testing Tools Directly

Before connecting to any LLM, we should test that our tools work correctly by calling them as regular Python functions.

In [ ]:
# Test the add tool
result = add(3, 5)
print(f"add(3, 5) = {result}")
assert result == 8, "add function is broken!"

# Test the multiply tool
result = multiply(4, 7)
print(f"multiply(4, 7) = {result}")
assert result == 28, "multiply function is broken!"

# Test the weather tool
result = get_weather("Istanbul")
print(f"\nget_weather('Istanbul') = {result}")
assert result["temp"] == 22

# Test with unknown city
result = get_weather("Mars")
print(f"get_weather('Mars') = {result}")
assert "error" in result

print("\nAll tool tests passed!")

## 6. Testing with the MCP Client

Now let's test our server using the MCP client SDK, which simulates how an LLM would interact with it.

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def test_mcp_server():
    """Connect to our MCP server and test it."""
    server_params = StdioServerParameters(
        command="python",
        args=["mcp_server.py"],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Initialize the connection
            await session.initialize()

            # List available tools
            tools = await session.list_tools()
            print("Available tools:")
            for tool in tools.tools:
                print(f"  - {tool.name}: {tool.description}")

            print("\n--- Calling 'add' tool ---")
            result = await session.call_tool("add", arguments={"a": 10, "b": 32})
            print(f"Result: {result.content[0].text}")

            print("\n--- Calling 'get_weather' tool ---")
            result = await session.call_tool("get_weather", arguments={"city": "Istanbul"})
            print(f"Result: {result.content[0].text}")

            # List available resources
            resources = await session.list_resources()
            print("\nAvailable resources:")
            for resource in resources.resources:
                print(f"  - {resource.uri}: {resource.name}")

            # Read a resource
            print("\n--- Reading 'config://app' resource ---")
            content = await session.read_resource("config://app")
            print(f"Content: {content.contents[0].text}")

# Run the test
await test_mcp_server()

## 7. Adding Input Validation to Tools

Good MCP tools should validate their inputs and return helpful error messages.

In [ ]:
from mcp.server.fastmcp import FastMCP
from typing import Annotated

mcp2 = FastMCP("ValidatedServer")

@mcp2.tool()
def divide(
    numerator: Annotated[float, "The number to divide"],
    denominator: Annotated[float, "The number to divide by (cannot be zero)"]
) -> float:
    """Divide the numerator by the denominator."""
    if denominator == 0:
        raise ValueError("Cannot divide by zero!")
    return numerator / denominator

# Test valid input
print(f"divide(10, 3) = {divide(10, 3):.4f}")

# Test invalid input
try:
    divide(10, 0)
except ValueError as e:
    print(f"Caught expected error: {e}")

# Show the tool's parameter descriptions using Annotated
print("\nTool info:")
for name, tool in mcp2._tool_manager._tools.items():
    print(f"  Tool: {name}")
    print(f"  Description: {tool.description}")
    print(f"  Parameters: {tool.parameters}")

## 8. Building a More Practical Tool

Let's build something more useful — a tool that can search and filter data.

In [ ]:
from mcp.server.fastmcp import FastMCP
from typing import Optional

mcp3 = FastMCP("FlightServer")

# Simulated flight database
FLIGHTS = [
    {"id": "TK101", "from": "Istanbul", "to": "London", "price": 350, "airline": "Turkish Airlines"},
    {"id": "TK202", "from": "Istanbul", "to": "Tokyo", "price": 800, "airline": "Turkish Airlines"},
    {"id": "BA303", "from": "London", "to": "New York", "price": 600, "airline": "British Airways"},
    {"id": "AA404", "from": "New York", "to": "Tokyo", "price": 1100, "airline": "American Airlines"},
    {"id": "TK505", "from": "Istanbul", "to": "New York", "price": 700, "airline": "Turkish Airlines"},
]

@mcp3.tool()
def search_flights(
    origin: Optional[str] = None,
    destination: Optional[str] = None,
    max_price: Optional[float] = None
) -> list[dict]:
    """Search for flights with optional filters for origin, destination, and max price."""
    results = FLIGHTS
    if origin:
        results = [f for f in results if f["from"].lower() == origin.lower()]
    if destination:
        results = [f for f in results if f["to"].lower() == destination.lower()]
    if max_price is not None:
        results = [f for f in results if f["price"] <= max_price]
    return results

@mcp3.tool()
def get_flight_details(flight_id: str) -> dict:
    """Get detailed information about a specific flight by its ID."""
    for flight in FLIGHTS:
        if flight["id"] == flight_id.upper():
            return flight
    return {"error": f"Flight {flight_id} not found"}

@mcp3.resource("data://flights/all")
def get_all_flights() -> str:
    """Return all available flights."""
    lines = [f"{f['id']}: {f['from']} -> {f['to']} (${f['price']})" for f in FLIGHTS]
    return "\n".join(lines)

# Test the search tool
print("All flights from Istanbul:")
for f in search_flights(origin="Istanbul"):
    print(f"  {f['id']}: {f['from']} -> {f['to']} ${f['price']}")

print("\nFlights under $500:")
for f in search_flights(max_price=500):
    print(f"  {f['id']}: {f['from']} -> {f['to']} ${f['price']}")

print("\nFlight TK202:")
print(f"  {get_flight_details('TK202')}")

## 9. Resource Templates

Resources can also be dynamic using URI templates. This lets you expose parameterized data.

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp4 = FastMCP("ResourceTemplateServer")

EMPLOYEES = {
    "1": {"name": "Alice", "role": "Engineer", "department": "Platform"},
    "2": {"name": "Bob", "role": "Designer", "department": "Product"},
    "3": {"name": "Charlie", "role": "Manager", "department": "Platform"},
}

@mcp4.resource("employee://{employee_id}")
def get_employee(employee_id: str) -> str:
    """Get employee information by ID."""
    emp = EMPLOYEES.get(employee_id)
    if emp:
        return f"Name: {emp['name']}\nRole: {emp['role']}\nDepartment: {emp['department']}"
    return f"Employee {employee_id} not found"

# Test the resource template
print("Employee 1:")
print(get_employee("1"))
print("\nEmployee 2:")
print(get_employee("2"))
print("\nEmployee 99 (not found):")
print(get_employee("99"))

---
## YOUR TURN: Exercise A

Create a new MCP server with the following:

1. A tool called `search_hotels` that accepts `city` (required) and `max_price` (optional) and returns matching hotels from a hardcoded list.
2. A tool called `book_hotel` that accepts `hotel_name` and `nights` and returns a booking confirmation with total price.
3. A resource at `data://hotels/cities` that lists all available cities.

Test each component with assertions.

In [ ]:
# YOUR TURN: Build the hotel MCP server

from mcp.server.fastmcp import FastMCP
from typing import Optional

hotel_mcp = FastMCP("HotelServer")

HOTELS = [
    {"name": "Grand Palace", "city": "Istanbul", "price_per_night": 120},
    {"name": "Bosphorus View", "city": "Istanbul", "price_per_night": 200},
    {"name": "The Shard Hotel", "city": "London", "price_per_night": 300},
    {"name": "Sakura Inn", "city": "Tokyo", "price_per_night": 150},
    {"name": "Central Park Lodge", "city": "New York", "price_per_night": 250},
]

# TODO: Define search_hotels tool

# TODO: Define book_hotel tool

# TODO: Define the data://hotels/cities resource

# TODO: Test your tools with assertions
# Example assertions to verify:
# assert len(search_hotels(city="Istanbul")) == 2
# assert len(search_hotels(city="Istanbul", max_price=150)) == 1
# booking = book_hotel(hotel_name="Grand Palace", nights=3)
# assert booking["total_price"] == 360

---
## YOUR TURN: Exercise B

Create a **complete MCP server script** that could be used as a real tool backend. It should have:

1. At least 3 tools related to a domain of your choice (e.g., task management, inventory, recipes)
2. At least 1 resource with a URI template
3. Proper input validation with helpful error messages
4. Write it as a standalone `.py` file using `%%writefile`

In [ ]:
%%writefile my_mcp_server.py
# YOUR TURN: Build a complete MCP server
# Choose a domain and implement tools + resources

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("MyServer")

# TODO: Define your data store

# TODO: Define at least 3 tools

# TODO: Define at least 1 resource with URI template

if __name__ == "__main__":
    mcp.run()

## 10. Running Your MCP Server

In a real setup, you would run your server from the terminal:

```bash
# Run with stdio transport (for local tools like Claude Desktop)
python mcp_server.py

# Or use the MCP CLI inspector to test interactively
mcp dev mcp_server.py

# Install in Claude Desktop
mcp install mcp_server.py
```

The `mcp dev` command opens an interactive inspector where you can:
- See all registered tools and resources
- Call tools with custom arguments
- Read resources
- Debug issues

## Key Takeaways

- **MCP** standardizes how LLMs connect to external tools and data
- **Tools** (`@mcp.tool()`) are functions the LLM can call — they DO things
- **Resources** (`@mcp.resource()`) expose data the LLM can READ
- Type hints and docstrings are critical — they tell the LLM what your tool does
- Always validate inputs and return clear error messages
- Test tools as regular functions before deploying as an MCP server

**Next:** In Exercise 2, we will build our first LangChain chain and see how to connect LLMs to structured workflows.